<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Logging, captura de errores y Organizador Inteligente de Documentos**

En los laboratorios anteriores hemos aprendido a:

- representar procesos;
- trabajar con estructuras;
- diseñar funciones;
- modularizar;
- manipular rutas;
- copiar y mover archivos;
- generar backups;
- planificar operaciones;
- validar integridad.

Ahora añadiremos la capa que convierte una automatización en un proceso **observable y robusto**:

> **logging + gestión de errores + continuidad segura**

Y cerraremos con la práctica integradora:

> **Organizador Inteligente de Documentos**

## Objetivos

Al finalizar deberías poder:

1. Explicar por qué `print()` no sustituye a logging.
2. Utilizar niveles `DEBUG`, `INFO`, `WARNING`, `ERROR`, `CRITICAL`.
3. Configurar logs en consola y fichero.
4. Crear un logger reusable.
5. Registrar contexto útil.
6. Capturar excepciones concretas.
7. Evitar que un único archivo detenga todo el lote.
8. Registrar éxitos y fallos.
9. Construir un resumen final.
10. Mantener trazabilidad.
11. Integrar backup, clasificación, movimiento y logging.
12. Construir un Organizador Inteligente de Documentos completo.

## 1. Preparación

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import logging
import shutil
import time
import re

import pandas as pd
import matplotlib.pyplot as plt

print("Entorno preparado.")

# 2. Crear entorno de práctica

In [ ]:
raiz = Path(
    "demo_organizador_logging"
)

if raiz.exists():
    shutil.rmtree(
        raiz
    )

entrada = raiz / "entrada"
salida = raiz / "salida"
backup = raiz / "backup"
logs = raiz / "logs"

for carpeta in [
    entrada,
    salida,
    backup,
    logs
]:
    carpeta.mkdir(
        parents=True,
        exist_ok=True
    )

print(
    "Entorno:",
    raiz.resolve()
)

# 3. Crear documentos de prueba

In [ ]:
documentos = {
    entrada / "Factura 001.PDF":
        "factura uno",

    entrada / "factura_002.pdf":
        "factura dos",

    entrada / "clientes.xlsx":
        "excel demo",

    entrada / "ventas.csv":
        "ventas,100",

    entrada / "foto evento.JPG":
        "imagen",

    entrada / "README":
        "sin extension",

    entrada / "archivo.zip":
        "zip demo",

    entrada / "documento.tmp":
        "tmp demo"
}

for ruta, contenido in documentos.items():
    ruta.write_text(
        contenido,
        encoding="utf-8"
    )

print(
    "Archivos:",
    len(
        documentos
    )
)

# 4. `print()` como primera aproximación

In [ ]:
for archivo in entrada.iterdir():
    if archivo.is_file():
        print(
            "Procesando:",
            archivo.name
        )

### Problema

`print()`:

- no tiene niveles;
- no añade timestamps automáticamente;
- no persiste por defecto;
- no distingue información de error;
- no escala bien a procesos grandes.

Por eso utilizaremos `logging`.

# 5. Primer logger

In [ ]:
logger = logging.getLogger(
    "organizador_demo"
)

logger.setLevel(
    logging.DEBUG
)

logger.handlers.clear()

handler = logging.StreamHandler()

formato = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s"
)

handler.setFormatter(
    formato
)

logger.addHandler(
    handler
)

logger.info(
    "Logger configurado."
)

# 6. Niveles de logging

In [ ]:
logger.debug(
    "Mensaje DEBUG"
)

logger.info(
    "Mensaje INFO"
)

logger.warning(
    "Mensaje WARNING"
)

logger.error(
    "Mensaje ERROR"
)

logger.critical(
    "Mensaje CRITICAL"
)

### Interpretación

Usaremos normalmente:

- `DEBUG`: detalle técnico;
- `INFO`: funcionamiento normal;
- `WARNING`: anomalía recuperable;
- `ERROR`: operación fallida;
- `CRITICAL`: problema grave del sistema.

# 7. Añadir log a fichero

In [ ]:
ruta_log = (
    logs
    / "organizador.log"
)

file_handler = logging.FileHandler(
    ruta_log,
    encoding="utf-8"
)

file_handler.setFormatter(
    formato
)

logger.addHandler(
    file_handler
)

logger.info(
    "Logging en fichero activado."
)

# 7.1 Inspeccionar fichero

In [ ]:
print(
    ruta_log.read_text(
        encoding="utf-8"
    )
)

# 8. Función reusable para crear logger

In [ ]:
def crear_logger(
    nombre,
    ruta_log,
    nivel=logging.INFO
):
    logger = logging.getLogger(
        nombre
    )

    logger.setLevel(
        nivel
    )

    logger.handlers.clear()

    formato = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    consola = logging.StreamHandler()
    consola.setFormatter(
        formato
    )

    fichero = logging.FileHandler(
        ruta_log,
        encoding="utf-8"
    )

    fichero.setFormatter(
        formato
    )

    logger.addHandler(
        consola
    )

    logger.addHandler(
        fichero
    )

    return logger


logger = crear_logger(
    "organizador",
    logs / "proceso.log",
    logging.DEBUG
)

logger.info(
    "Logger reusable listo."
)

# 9. Provocar un error de archivo inexistente

In [ ]:
archivo_inexistente = (
    entrada
    / "no_existe.pdf"
)

try:
    contenido = (
        archivo_inexistente
        .read_text(
            encoding="utf-8"
        )
    )

except FileNotFoundError as error:
    logger.error(
        "Archivo no encontrado: %s",
        archivo_inexistente
    )

    print(
        type(error).__name__
    )

### Interpretación

Capturamos una excepción **concreta**.

Evitar `except Exception:` indiscriminadamente mejora el diagnóstico.

# 10. Error por permisos: simulación conceptual

En un sistema real pueden aparecer:

- `PermissionError`;
- `FileNotFoundError`;
- `FileExistsError`;
- errores de lectura;
- errores de escritura;
- archivos bloqueados;
- rutas inválidas.

Nuestro pipeline debe decidir qué errores:

- detienen todo;
- se registran y permiten continuar.

# 11. Función de hash

In [ ]:
def calcular_hash(
    ruta
):
    hasher = hashlib.sha256()

    with Path(
        ruta
    ).open(
        "rb"
    ) as fichero:

        while True:
            bloque = fichero.read(
                8192
            )

            if not bloque:
                break

            hasher.update(
                bloque
            )

    return hasher.hexdigest()

# 12. Función de normalización

In [ ]:
def normalizar_nombre(
    nombre
):
    ruta = Path(
        nombre
    )

    extension = (
        ruta.suffix.lower()
        if ruta.suffix
        else ""
    )

    stem = (
        ruta.stem
        .strip()
        .lower()
    )

    stem = re.sub(
        r"\s+",
        "_",
        stem
    )

    stem = re.sub(
        r"[^a-z0-9_-]+",
        "",
        stem
    )

    return (
        f"{stem}{extension}"
        if extension
        else stem
    )

# 13. Reglas de clasificación

In [ ]:
REGLAS = {
    ".pdf": "PDF",
    ".xlsx": "Excel",
    ".csv": "Datos",
    ".jpg": "Imagenes",
    ".jpeg": "Imagenes",
    ".png": "Imagenes"
}


def clasificar_archivo(
    ruta
):
    ruta = Path(
        ruta
    )

    extension = (
        ruta.suffix.lower()
        if ruta.suffix
        else "sin_extension"
    )

    if extension == "sin_extension":
        return "Revision"

    return REGLAS.get(
        extension,
        "Revision"
    )

# 14. Función de destino único

In [ ]:
def destino_unico(
    destino
):
    destino = Path(
        destino
    )

    if not destino.exists():
        return destino

    contador = 1

    while True:
        candidato = (
            destino
            .with_name(
                f"{destino.stem}_{contador}{destino.suffix}"
            )
        )

        if not candidato.exists():
            return candidato

        contador += 1

# 15. Crear plan con logging

In [ ]:
def construir_plan(
    carpeta_entrada,
    carpeta_salida,
    logger
):
    filas = []

    for ruta in Path(
        carpeta_entrada
    ).rglob("*"):

        if not ruta.is_file():
            continue

        categoria = (
            clasificar_archivo(
                ruta
            )
        )

        nombre = normalizar_nombre(
            ruta.name
        )

        destino = (
            Path(
                carpeta_salida
            )
            / categoria
            / nombre
        )

        logger.debug(
            "Planificando %s -> %s",
            ruta,
            destino
        )

        filas.append({
            "origen":
                str(
                    ruta
                ),

            "categoria":
                categoria,

            "destino_propuesto":
                str(
                    destino
                )
        })

    return pd.DataFrame(
        filas
    )


plan = construir_plan(
    entrada,
    salida,
    logger
)

plan

# 16. Visualizar plan

In [ ]:
plan[
    "categoria"
].value_counts()

In [ ]:
conteo = (
    plan[
        "categoria"
    ]
    .value_counts()
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    conteo.index,
    conteo.values
)

plt.ylabel(
    "Archivos"
)

plt.title(
    "Plan por categoría"
)

plt.tight_layout()
plt.show()

# 17. Crear backup con logging

In [ ]:
def crear_backup(
    carpeta_entrada,
    carpeta_backup,
    logger
):
    marca = time.strftime(
        "%Y%m%d_%H%M%S"
    )

    base = (
        Path(
            carpeta_backup
        )
        / f"entrada_{marca}"
    )

    logger.info(
        "Creando backup de %s",
        carpeta_entrada
    )

    archivo = shutil.make_archive(
        str(
            base
        ),
        "zip",
        root_dir=str(
            carpeta_entrada
        )
    )

    logger.info(
        "Backup creado: %s",
        archivo
    )

    return Path(
        archivo
    )


ruta_backup = crear_backup(
    entrada,
    backup,
    logger
)

print(
    ruta_backup
)

# 18. Procesar un archivo de forma robusta

In [ ]:
def mover_archivo_seguro(
    origen,
    destino,
    logger
):
    origen = Path(
        origen
    )

    destino = Path(
        destino
    )

    try:
        if not origen.exists():
            raise FileNotFoundError(
                origen
            )

        destino.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        destino_final = (
            destino_unico(
                destino
            )
        )

        hash_antes = calcular_hash(
            origen
        )

        shutil.move(
            str(
                origen
            ),
            str(
                destino_final
            )
        )

        hash_despues = calcular_hash(
            destino_final
        )

        if hash_antes != hash_despues:
            raise RuntimeError(
                "El hash cambió tras mover."
            )

        logger.info(
            "Movido: %s -> %s",
            origen,
            destino_final
        )

        return {
            "origen":
                str(
                    origen
                ),

            "destino":
                str(
                    destino_final
                ),

            "estado":
                "ok",

            "error":
                None
        }

    except FileNotFoundError as error:
        logger.error(
            "No existe origen: %s",
            origen
        )

        return {
            "origen":
                str(
                    origen
                ),

            "destino":
                str(
                    destino
                ),

            "estado":
                "error",

            "error":
                str(
                    error
                )
        }

    except PermissionError as error:
        logger.error(
            "Permiso denegado: %s",
            origen
        )

        return {
            "origen":
                str(
                    origen
                ),

            "destino":
                str(
                    destino
                ),

            "estado":
                "error",

            "error":
                str(
                    error
                )
        }

    except Exception as error:
        logger.exception(
            "Error inesperado procesando %s",
            origen
        )

        return {
            "origen":
                str(
                    origen
                ),

            "destino":
                str(
                    destino
                ),

            "estado":
                "error",

            "error":
                str(
                    error
                )
        }

# 19. Provocar error y comprobar continuidad

In [ ]:
prueba_error = mover_archivo_seguro(
    entrada
    / "archivo_que_no_existe.pdf",

    salida
    / "PDF"
    / "archivo_que_no_existe.pdf",

    logger
)

prueba_error

### Interpretación

El error queda:

- registrado;
- estructurado;
- devuelto.

Y el programa puede continuar con el resto del lote.

# 20. Ejecutar lote completo

In [ ]:
def ejecutar_plan(
    plan,
    logger
):
    resultados = []

    logger.info(
        "Inicio de ejecución. Archivos: %d",
        len(
            plan
        )
    )

    for _, fila in plan.iterrows():

        resultado = (
            mover_archivo_seguro(
                fila[
                    "origen"
                ],
                fila[
                    "destino_propuesto"
                ],
                logger
            )
        )

        resultado[
            "categoria"
        ] = fila[
            "categoria"
        ]

        resultados.append(
            resultado
        )

    logger.info(
        "Fin de ejecución."
    )

    return pd.DataFrame(
        resultados
    )


resultado = ejecutar_plan(
    plan,
    logger
)

resultado

# 21. Validar continuidad

In [ ]:
print(
    "Procesados:",
    len(
        resultado
    )
)

print(
    "Esperados:",
    len(
        plan
    )
)

assert (
    len(
        resultado
    )
    == len(
        plan
    )
)

# 22. Resumen de éxitos y errores

In [ ]:
resumen_estado = (
    resultado[
        "estado"
    ]
    .value_counts()
)

resumen_estado

In [ ]:
plt.figure(
    figsize=(7, 5)
)

plt.bar(
    resumen_estado.index,
    resumen_estado.values
)

plt.ylabel(
    "Archivos"
)

plt.title(
    "Estado final del procesamiento"
)

plt.tight_layout()
plt.show()

# 23. Revisar errores

In [ ]:
errores = resultado[
    resultado[
        "estado"
    ] == "error"
]

errores

### Interpretación

Los errores no desaparecen.

Quedan disponibles para:

- auditoría;
- reintento;
- revisión humana.

# 24. KPIs del proceso

In [ ]:
kpis = {
    "total":
        len(
            resultado
        ),

    "ok":
        int(
            (
                resultado[
                    "estado"
                ]
                == "ok"
            ).sum()
        ),

    "error":
        int(
            (
                resultado[
                    "estado"
                ]
                == "error"
            ).sum()
        ),

    "revision":
        int(
            (
                resultado[
                    "categoria"
                ]
                == "Revision"
            ).sum()
        )
}

kpis[
    "tasa_exito"
] = (
    kpis[
        "ok"
    ]
    / kpis[
        "total"
    ]
    if kpis[
        "total"
    ]
    else 0
)

kpis

# 25. Guardar auditoría

In [ ]:
ruta_auditoria = (
    logs
    / "auditoria.csv"
)

resultado.to_csv(
    ruta_auditoria,
    index=False
)

print(
    ruta_auditoria.exists()
)

# 26. Leer log del proceso

In [ ]:
contenido_log = (
    logs
    / "proceso.log"
).read_text(
    encoding="utf-8"
)

print(
    contenido_log[-3000:]
)

### Interpretación

Ya tenemos dos capas de trazabilidad:

1. **log textual**;
2. **auditoría estructurada CSV**.

# 27. Filtrar log por nivel de forma simple

In [ ]:
lineas_error = [
    linea
    for linea in contenido_log.splitlines()
    if "ERROR" in linea
]

lineas_error[:10]

# 28. Crear una segunda bandeja para la práctica final

In [ ]:
entrada_final = (
    raiz
    / "entrada_final"
)

entrada_final.mkdir(
    exist_ok=True
)

documentos_finales = {
    entrada_final / "Contrato Cliente A.PDF":
        "contrato",

    entrada_final / "ventas septiembre.csv":
        "ventas",

    entrada_final / "foto.PNG":
        "imagen",

    entrada_final / "notas.TXT":
        "texto",

    entrada_final / "archivo":
        "sin extension",

    entrada_final / "paquete.zip":
        "zip"
}

for ruta, contenido in documentos_finales.items():
    ruta.write_text(
        contenido,
        encoding="utf-8"
    )

print(
    [
        x.name
        for x
        in entrada_final.iterdir()
    ]
)

# 29. Organizador Inteligente de Documentos — versión Módulo 1

En este punto “inteligente” significa que el sistema:

- inspecciona;
- clasifica mediante reglas;
- detecta desconocidos;
- normaliza;
- evita colisiones;
- crea backup;
- mueve;
- valida;
- registra;
- deriva casos dudosos a revisión.

La IA generativa llegará más adelante.

# 30. Función integradora

In [ ]:
def organizar_documentos(
    carpeta_entrada,
    carpeta_salida,
    carpeta_backup,
    logger
):
    carpeta_entrada = Path(
        carpeta_entrada
    )

    carpeta_salida = Path(
        carpeta_salida
    )

    carpeta_backup = Path(
        carpeta_backup
    )

    logger.info(
        "=== NUEVA EJECUCIÓN ==="
    )

    if not carpeta_entrada.exists():
        raise FileNotFoundError(
            carpeta_entrada
        )

    plan = construir_plan(
        carpeta_entrada,
        carpeta_salida,
        logger
    )

    logger.info(
        "Plan generado: %d archivos",
        len(
            plan
        )
    )

    backup_creado = crear_backup(
        carpeta_entrada,
        carpeta_backup,
        logger
    )

    resultados = ejecutar_plan(
        plan,
        logger
    )

    metricas = {
        "total":
            len(
                resultados
            ),

        "ok":
            int(
                (
                    resultados[
                        "estado"
                    ]
                    == "ok"
                ).sum()
            ),

        "error":
            int(
                (
                    resultados[
                        "estado"
                    ]
                    == "error"
                ).sum()
            ),

        "revision":
            int(
                (
                    resultados[
                        "categoria"
                    ]
                    == "Revision"
                ).sum()
            )
    }

    return {
        "plan":
            plan,

        "resultados":
            resultados,

        "metricas":
            metricas,

        "backup":
            backup_creado
    }

# 31. Ejecutar práctica integradora

In [ ]:
salida_final = (
    raiz
    / "salida_final"
)

backup_final = (
    raiz
    / "backup_final"
)

salida_final.mkdir(
    exist_ok=True
)

backup_final.mkdir(
    exist_ok=True
)

ejecucion_final = organizar_documentos(
    entrada_final,
    salida_final,
    backup_final,
    logger
)

ejecucion_final[
    "metricas"
]

# 32. Ver plan final

In [ ]:
ejecucion_final[
    "plan"
]

# 33. Ver resultados finales

In [ ]:
ejecucion_final[
    "resultados"
]

# 34. Visualizar categorías finales

In [ ]:
categorias_final = (
    ejecucion_final[
        "resultados"
    ][
        "categoria"
    ]
    .value_counts()
)

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    categorias_final.index,
    categorias_final.values
)

plt.ylabel(
    "Archivos"
)

plt.title(
    "Organizador documental: resultado final"
)

plt.tight_layout()
plt.show()

# 35. Ver árbol final

In [ ]:
for ruta in sorted(
    salida_final.rglob("*")
):
    nivel = (
        len(
            ruta.relative_to(
                salida_final
            ).parts
        )
        - 1
    )

    sangria = (
        "    " * nivel
    )

    simbolo = (
        "📁"
        if ruta.is_dir()
        else "📄"
    )

    print(
        f"{sangria}{simbolo} {ruta.name}"
    )

# 36. Validaciones finales

In [ ]:
resultado_final = (
    ejecucion_final[
        "resultados"
    ]
)

assert (
    len(
        resultado_final
    )
    == len(
        documentos_finales
    )
)

assert resultado_final[
    "destino"
].apply(
    lambda x:
        Path(
            x
        ).exists()
).all()

assert ejecucion_final[
    "backup"
].exists()

print(
    "Organizador final validado."
)

# 37. Qué queda pendiente para IA

In [ ]:
casos_revision = (
    resultado_final[
        resultado_final[
            "categoria"
        ]
        == "Revision"
    ]
)

casos_revision

### Interpretación

Estos archivos son candidatos futuros para una segunda capa:

```text
reglas no resuelven
        ↓
extraer contenido
        ↓
LLM / clasificación
        ↓
confianza
        ↓
clasificar o revisión humana
```

Así conectaremos el Módulo 1 con el Módulo 3.

# 38. Checklist de robustez

- [ ] ¿El proceso genera logs?
- [ ] ¿Hay timestamps?
- [ ] ¿Distinguimos INFO/WARNING/ERROR?
- [ ] ¿Los errores no desaparecen?
- [ ] ¿Un fallo aislado permite continuar?
- [ ] ¿Se registra archivo origen?
- [ ] ¿Se registra destino?
- [ ] ¿Existe backup?
- [ ] ¿Existe auditoría estructurada?
- [ ] ¿Los casos desconocidos van a revisión?
- [ ] ¿Se valida integridad?
- [ ] ¿Se calculan KPIs?
- [ ] ¿El proceso puede auditarse después?

# 39. Ejercicios finales

### Ejercicio 1 — Logger
Configura consola y fichero.

### Ejercicio 2 — Niveles
Genera ejemplos de DEBUG, INFO, WARNING, ERROR y CRITICAL.

### Ejercicio 3 — FileNotFoundError
Provoca y captura un archivo inexistente.

### Ejercicio 4 — Contexto
Registra nombre de archivo y operación.

### Ejercicio 5 — Batch
Haz que un error no detenga el lote.

### Ejercicio 6 — Resultado estructurado
Devuelve estado y mensaje de error.

### Ejercicio 7 — Backup
Registra inicio y fin del backup.

### Ejercicio 8 — Movimiento
Registra origen y destino.

### Ejercicio 9 — Hash
Provoca una validación de integridad.

### Ejercicio 10 — Auditoría
Crea CSV de resultados.

### Ejercicio 11 — KPIs
Calcula tasa de éxito y revisión.

### Ejercicio 12 — Log analysis
Filtra las líneas `ERROR`.

### Ejercicio 13 — Reintento conceptual
Diseña cómo reintentarías únicamente los fallos.

### Ejercicio 14 — Organizador
Integra clasificación, normalización y movimiento.

### Ejercicio 15 — Revisión
Deriva extensiones desconocidas.

### Ejercicio 16 — Segunda ejecución
Comprueba que no sobrescribe.

### Ejercicio 17 — Modularización
Separa logger, clasificación y ejecución en módulos.

### Ejercicio 18 — Reto final
Construye tu propio **Organizador Inteligente de Documentos** con:

1. inventario;
2. reglas;
3. normalización;
4. plan;
5. backup;
6. movimientos seguros;
7. logging;
8. excepciones;
9. auditoría;
10. KPIs;
11. revisión humana;
12. validaciones finales.

# 40. Preguntas de reflexión

1. ¿Qué aporta logging frente a `print()`?
2. ¿Cuándo usarías WARNING?
3. ¿Cuándo usarías ERROR?
4. ¿Por qué capturar excepciones concretas?
5. ¿Cuándo debería detenerse todo el pipeline?
6. ¿Cuándo debería continuar?
7. ¿Qué información necesita un log útil?
8. ¿Qué diferencia hay entre log y auditoría CSV?
9. ¿Qué aporta un backup?
10. ¿Por qué guardar origen y destino?
11. ¿Qué significa trazabilidad?
12. ¿Por qué calcular KPIs?
13. ¿Qué archivos deben ir a revisión?
14. ¿Dónde incorporaremos IA más adelante?
15. ¿Qué hace que este organizador sea robusto?

# 41. Cierre del Módulo 1

Con este laboratorio cerramos el primer módulo.

La progresión completa ha sido:

```text
fundamentos de automatización
        ↓
estructuras de datos
        ↓
funciones y módulos
        ↓
pathlib / os / shutil / glob
        ↓
automatización real de archivos
        ↓
logging + errores
        ↓
Organizador Inteligente de Documentos
```

En el siguiente módulo cambiaremos de tipo de automatización y entraremos en:

```text
Excel
openpyxl
informes
APIs
web scraping
configuración
Git/GitHub
```